In [21]:
from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [22]:
import h5py
import numpy as np

try:
    import torch
except ImportError:
    torch = None


def load_item(h5_path: str, split: str, idx: int, as_torch: bool = False):
    """
    Read a single item from:
      pattern_{split}, params_{split}, neff_{split}, weight_{split}

    Returns a dict:
      {
        "pattern":  [H,W] or [1,H,W]  (numpy or torch),
        "params":   [P],
        "neff":     scalar,
        "weight":   scalar,
      }
    """
    if split not in ("train", "test"):
        raise ValueError("split must be 'train' or 'test'")

    patt_key   = f"pattern_{split}"
    params_key = f"params_{split}"
    neff_key   = f"neff_{split}"
    weight_key = f"weight_{split}"

    with h5py.File(h5_path, "r") as f:
        # sanity checks
        for k in (patt_key, params_key, neff_key, weight_key):
            if k not in f:
                raise KeyError(f"Missing dataset '{k}' in {h5_path}")

        pattern = f[patt_key][idx]       # [H,W] or [1,H,W]
        params  = f[params_key][idx]     # [P]
        neff    = f[neff_key][idx]       # scalar or shape ()
        weight  = f[weight_key][idx]     # scalar or shape ()

    # Normalize shapes/types a bit
    pattern = np.asarray(pattern)
    params  = np.asarray(params, dtype=np.float32)
    neff    = float(np.asarray(neff))
    weight  = float(np.asarray(weight))

    # If you prefer [1,H,W] consistently, uncomment:
    # if pattern.ndim == 2:
    #     pattern = pattern[None, :, :]

    if as_torch:
        if torch is None:
            raise RuntimeError("PyTorch not available; set as_torch=False or install torch.")
        pattern = torch.from_numpy(pattern).float()
        params  = torch.from_numpy(params).float()
        neff    = torch.tensor(neff, dtype=torch.float32)
        weight  = torch.tensor(weight, dtype=torch.float32)

    return {"pattern": pattern, "params": params, "neff": neff, "weight": weight}


In [23]:
item = load_item("new_top1_cnn.h5", split="train", idx=50, as_torch=False)
print(item["pattern"], item["params"], item["neff"], item["weight"])

# If you need the quartered pattern (and you have quarter()):
# from my_runtime_objects import quarter
# q = quarter(item["pattern"])


[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 1 1 1]
 [0 0 0 ... 1 1 1]
 [0 0 0 ... 1 1 1]] [5.9200001e-01 6.7567566e-07 2.8900001e+00 8.8400002e+00] 5.274226188659668 70.8381118774414


In [24]:
class CombinedModeWeightNet(nn.Module):
    def __init__(self, mode_model, weight_model):
        super().__init__()
        self.mode_model = mode_model
        self.weight_model = weight_model

    def forward(self, x_img, x_cond):
        mode = self.mode_model(x_img, x_cond)   # [B, 1]
        weight = self.weight_model(x_img, x_cond)  # [B, 1]
        return torch.cat((mode, weight), dim=1)   # [B, 2]

from only_mode_only_weight_v3 import No_normal_modewieght_net

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode0_model = No_normal_modewieght_net().to(device)
weight0_model = No_normal_modewieght_net().to(device)

mode0_model.load_state_dict(torch.load('models/only_first_mode_no_normalization_with_less_dropout.pth', map_location=torch.device(device)))
weight0_model.load_state_dict(torch.load('models/only_first_weight_no_normalization_with_less_dropout_100.pth', map_location=torch.device(device)))

cnn = CombinedModeWeightNet(mode0_model, weight0_model).to(device)
cnn.eval()

CombinedModeWeightNet(
  (mode_model): No_normal_modewieght_net(
    (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn2): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn3): GroupNorm(8, 256, eps=1e-05, affine=True)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv4): Conv2d(260, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn4): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn5): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv6): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn6): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv7): Conv2d(256, 256, kernel_size=(3, 3),

In [25]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [26]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        '''
        standard ResNet style convolutional block, for image processing
        '''
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            # this adds on correct residual in case channels have increased
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2


In [27]:
class UnetDown(nn.Module):
    """
    Downsampling path for U-Net, reduces spatial resolution while increasing feature depth
    Input: Image batch, size (batchsize, 1, 32, 32)
    Output: size (batchsize, out_channels, 16, 16)
    Output:
    """
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        '''
        process and downscale the image feature maps
        '''
        layers = [ResidualConvBlock(
            in_channels, out_channels), nn.MaxPool2d(2)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Doubles spatial dimensions, halves feature dimensions
        # My channel dimension for image will always be 1, greyscale
        return self.model(x)


In [28]:
class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        '''
        process and upscale the image feature maps
        Doubles spatial size, but decreases channels:
        input: 2 vectors of size (binsize, in_channels / 2, h, w) 
        output: (binsize, outchannels, 2h, 2w)
        '''
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        """
        x is the upsampled features from previous decoder layer
        skip is the skip connection from the encoder, same size as x
        """
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x

In [29]:
class EmbedFC(nn.Module):
    """
    Use FC layer for embedding 1-d metadata, like modes+weights
    (putting into higher dimension)
    Effectively our conditional
    input: Conditional, size (batchsize, input_dim = 4+4)
    Output: Higherdimensional tensor, size (batchsize, output_dim)
    
    """
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things  
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assumes you already have these:
# - ResidualConvBlock(in_ch, out_ch, is_res=True)
# - UnetDown(in_ch, out_ch)
# - UnetUp(in_ch, out_ch)
# - EmbedFC(in_dim, out_dim)

class ContextUnet(nn.Module):
    """
    U-Net for conditional image generation (32x32), where conditioning is
    provided by *tiled* scalar maps concatenated to the input channels.

    Conditioning (this version):
      - top (mode, weight): shape [B, 2]  (or [B, 1, 2], both supported)
      - params:             shape [B, 4]
      -> total 6 scalars per sample, each tiled to [H, W] and concatenated as channels.

    Timestep t is still embedded via an MLP and injected on the up path.
    """

    def __init__(self, in_channels=1, n_feat=256, use_time_embed=True):
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.use_time_embed = use_time_embed

        # 6 tiled condition channels (2 from top mode/weight + 4 params)
        self.cond_channels = 6
        lifted_in = in_channels + self.cond_channels

        # Encoder
        self.init_conv = ResidualConvBlock(lifted_in, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)          # 32x32 -> 16x16 (n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)      # 16x16 -> 8x8   (2*n_feat)

        # Latent pooling to 1x1
        self.to_vec = nn.Sequential(nn.AvgPool2d(8), nn.GELU())

        # Optional timestep embedding (kept as MLP, not tiled)
        if use_time_embed:
            self.timeembed1 = EmbedFC(1, 2 * n_feat)
            self.timeembed2 = EmbedFC(1, 1 * n_feat)

        # Decoder
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8),  # 1x1 -> 8x8
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        # Concats with skips: (2*n_feat up) + (2*n_feat skip) = 4*n_feat -> n_feat
        self.up1 = UnetUp(4 * n_feat, n_feat)          # 8x8 -> 16x16
        # (n_feat up) + (n_feat skip) = 2*n_feat -> n_feat
        self.up2 = UnetUp(2 * n_feat, n_feat)          # 16x16 -> 32x32

        # Output head: concat with early skip (x after init_conv) -> 2*n_feat
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    @staticmethod
    def _tile_condition(top_pair, params, H, W):
        """
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        Returns tiled tensor of shape [B, 6, H, W]
        """
        if top_pair.dim() == 3:
            # allow [B, 1, 2] → [B, 2]
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2] (mode, weight)"

        B = top_pair.shape[0]
        cond = torch.cat([top_pair, params], dim=1)    # [B, 6]
        cond = cond.unsqueeze(-1).unsqueeze(-1)        # [B, 6, 1, 1]
        cond = cond.repeat(1, 1, H, W)                 # [B, 6, H, W]
        return cond

    def forward(self, x, top_pair, params, t):
        """
        x:        [B, 1, 32, 32]  (noisy waveguide / latent)
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        t:        [B, 1] (scalar timestep per sample)
        """
        B, _, H, W = x.shape
        assert H == 32 and W == 32, "This U-Net assumes 32x32 spatial size."

        # Build and add tiled conditions as channels at the input
        cond_maps = self._tile_condition(top_pair, params, H, W)  # [B, 6, 32, 32]
        x_in = torch.cat([x, cond_maps], dim=1)                   # [B, 1+6, 32, 32]

        # Encoder
        x0 = self.init_conv(x_in)   # [B, n_feat, 32, 32]
        d1 = self.down1(x0)         # [B, n_feat, 16, 16]
        d2 = self.down2(d1)         # [B, 2*n_feat, 8, 8]
        h  = self.to_vec(d2)        # [B, 2*n_feat, 1, 1]

        # Decode (optionally inject time embeddings)
        up1 = self.up0(h)           # [B, 2*n_feat, 8, 8]

        if self.use_time_embed:
            temb1 = self.timeembed1(t).view(B, 2 * self.n_feat, 1, 1)
            temb2 = self.timeembed2(t).view(B, 1 * self.n_feat, 1, 1)
            up1 = up1 + temb1

        u2 = self.up1(up1, d2)      # -> [B, n_feat, 16, 16]
        if self.use_time_embed:
            u2 = u2 + temb2

        u3 = self.up2(u2, d1)       # -> [B, n_feat, 32, 32]

        # Final head with early skip (x0)
        out = self.out(torch.cat([u3, x0], dim=1))  # [B, 1, 32, 32]
        return out


In [31]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [32]:
import importlib
import waveguide_dataset_top1
importlib.reload(waveguide_dataset_top1)
from waveguide_dataset_top1 import WaveguideDatasetTop1

In [33]:
import torch
import torch.nn as nn
import numpy as np

class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model = nn_model.to(device)

        # register all schedule buffers
        sched = ddpm_schedules(betas[0], betas[1], n_T)
        for k, v in sched.items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    @staticmethod
    def _apply_mask(top_pair, params, context_mask):
        """
        context_mask: [B] or [B,1] of {0,1}, where 1 => drop conditioning.
        Returns masked copies (zeros when dropped).

        top_pair: [B, 2] or [B, 1, 2]
        params:   [B, 4]
        """
        if context_mask.dim() == 1:
            context_mask = context_mask.unsqueeze(1)  # [B,1]

        # Normalize top_pair shape to [B, 2]
        if top_pair.dim() == 3:
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2]"

        tp = top_pair * (1.0 - context_mask)            # [B,2] broadcast
        pr = params   * (1.0 - context_mask)            # [B,4] broadcast
        return tp, pr

    def forward(self, x, top_pair, params):
        """
        Training step:
          x:        [B,1,32,32] (clean image x0)
          top_pair: [B,2] (mode_0, weight_0)  or [B,1,2]
          params:   [B,4]
        """
        B = x.shape[0]
        _ts = torch.randint(1, self.n_T + 1, (B,), device=self.device)        # [B]
        noise = torch.randn_like(x)                                           # [B,1,32,32]

        x_t = (
            self.sqrtab[_ts, None, None, None] * x
            + self.sqrtmab[_ts, None, None, None] * noise
        )

        # classifier-free dropout
        context_mask = torch.bernoulli(
            torch.full((B,), self.drop_prob, device=self.device)
        )  # [B] in {0,1}

        tp_masked, pr_masked = self._apply_mask(top_pair, params, context_mask)

        # normalized timestep as [B,1]
        t_norm = (_ts.float() / self.n_T).unsqueeze(1)  # [B,1]

        pred_noise = self.nn_model(x_t, tp_masked, pr_masked, t_norm)
        return self.loss_mse(noise, pred_noise)

    @torch.no_grad()
    def sample(self, n_sample, size, device, top_pair, params, guide_w=0.0):
        """
        Sampling with classifier-free guidance (CFG).

        Args
        ----
        n_sample: int
        size:     tuple like (1, 32, 32)
        device:   torch.device
        top_pair: [n_sample, 2] tensor (mode_0, weight_0)  or [n_sample, 1, 2]
        params:   [n_sample, 4] tensor
        guide_w:  float guidance scale (0 = no CFG)

        Returns
        -------
        x_T->x_0 sample tensor [n_sample, 1, 32, 32], and numpy trajectory.
        """
        assert top_pair.shape[0] == n_sample and params.shape[0] == n_sample

        x_i = torch.randn(n_sample, *size, device=device)

        # Build masks for double batch (first half conditioned, second half dropped)
        context_mask_cond   = torch.zeros(n_sample, device=device)  # keep
        context_mask_uncond = torch.ones(n_sample,  device=device)  # drop

        # Precompute cond/uncond views
        tp_cond, pr_cond       = self._apply_mask(top_pair, params, context_mask_cond)
        tp_uncond, pr_uncond   = self._apply_mask(top_pair, params, context_mask_uncond)

        x_i_store = []
        for i in range(self.n_T, 0, -1):
            # timestep scalar normalized
            t_norm = torch.full((n_sample, 1), i / self.n_T, device=device)

            # double the batch (conditioned + unconditioned)
            x_in = torch.cat([x_i, x_i], dim=0)
            t_in = torch.cat([t_norm, t_norm], dim=0)             # [2B,1]
            tp_in = torch.cat([tp_cond, tp_uncond], dim=0)        # [2B,2]
            pr_in = torch.cat([pr_cond, pr_uncond], dim=0)        # [2B,4]

            # predict noise for both halves
            eps = self.nn_model(x_in, tp_in, pr_in, t_in)         # [2B,1,32,32]
            eps1, eps2 = eps[:n_sample], eps[n_sample:]           # cond, uncond

            # CFG combine
            eps_cfg = (1 + guide_w) * eps1 - guide_w * eps2

            z = torch.randn_like(x_i) if i > 1 else 0.0
            x_i = (
                self.oneover_sqrta[i] * (x_i - eps_cfg * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )

            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        x_i_store = np.array(x_i_store)
        return x_i, x_i_store


In [34]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ----------------------------------------------------------------------
# Eval on a subset of the test loader with the same gamma-blended loss
# ----------------------------------------------------------------------
@torch.no_grad()
def evaluate_on_test_portion(
    ddpm,
    test_loader,
    n_T: int,
    device: torch.device,
    cnn=None,
    test_eval_fraction: float = 0.25,
    clamp_for_cnn: bool = True,
    gamma: float = 0.5,
):
    """
    Blended loss:
      loss = gamma * MSE(x0_hat, x_real)  +  (1 - gamma) * MSE(cnn(x0_hat, params), cond)

    Notes
    -----
    - If gamma == 1.0, CNN is not used (even if provided).
    - If gamma < 1.0, cnn must be provided.
    """
    ddpm.eval()
    was = ddpm.drop_prob
    ddpm.drop_prob = 0.0

    total, count = 0.0, 0
    max_batches = max(1, int(np.ceil(test_eval_fraction * len(test_loader)))) if len(test_loader) > 0 else 1
    use_cnn_branch = (gamma < 1.0)

    for b_idx, (cond, params, x_real) in enumerate(test_loader):
        x_real = x_real.to(device, non_blocking=True)  # [B,1,32,32]
        cond   = cond.to(device, non_blocking=True)    # [B,2] (neff, weight)
        params = params.to(device, non_blocking=True)  # [B,P]

        B = x_real.size(0)
        _ts   = torch.randint(1, n_T + 1, (B,), device=device)
        noise = torch.randn_like(x_real)
        x_t   = ddpm.sqrtab[_ts, None, None, None] * x_real + ddpm.sqrtmab[_ts, None, None, None] * noise
        t_norm = (_ts.float() / n_T).unsqueeze(1)

        pred_noise = ddpm.nn_model(x_t, cond, params, t_norm)
        x0_hat = (x_t - ddpm.sqrtmab[_ts, None, None, None] * pred_noise) / ddpm.sqrtab[_ts, None, None, None]
        if clamp_for_cnn:
            x0_hat = x0_hat.clamp(0.0, 1.0)

        # image similarity
        loss_img = F.mse_loss(x0_hat, x_real)

        if use_cnn_branch:
            assert cnn is not None, "gamma < 1.0 requires a cnn for the target-in-CNN-space."
            pred_top = cnn(x0_hat, params)  # [B,2]
            targ_top = cond                 # dataset ground-truth [neff, weight]
            loss_cnn = F.mse_loss(pred_top, targ_top)
        else:
            loss_cnn = x_real.new_tensor(0.0)

        loss = gamma * loss_img + (1.0 - gamma) * loss_cnn
        total += float(loss.item())
        count += 1
        if b_idx + 1 >= max_batches:
            break

    ddpm.drop_prob = was
    return total / max(count, 1)


# ----------------------------------------------------------------------
# Final GIF saving (unchanged except: pass cond directly)
# ----------------------------------------------------------------------
@torch.no_grad()
def save_final_gifs(
    ddpm,
    test_loader,
    test_ds,
    device: torch.device,
    ws_test=(0.0, 0.5, 2.0),
    max_samples: int = 32,
    gif_prefix: str = "gif_final",
):
    try:
        cond_eval, params_eval, x_real_batch = next(iter(test_loader))
    except StopIteration:
        fallback_loader = DataLoader(test_ds, batch_size=min(max_samples, len(test_ds)))
        cond_eval, params_eval, x_real_batch = next(iter(fallback_loader))

    cond_eval    = cond_eval.to(device)
    params_eval  = params_eval.to(device)
    n_sample = min(max_samples, cond_eval.shape[0])
    cond_eval    = cond_eval[:n_sample]
    params_eval  = params_eval[:n_sample]

    for w in ws_test:
        x_gen, x_gen_store = ddpm.sample(
            n_sample=n_sample,
            size=(1, 32, 32),
            device=device,
            top_pair=cond_eval,   # [B,2] = (neff, weight)
            params=params_eval,
            guide_w=w,
        )

        fig, axs = plt.subplots(
            nrows=int(n_sample // 8) if n_sample >= 8 else 1,
            ncols=min(8, n_sample),
            sharex=True, sharey=True, figsize=(8, 3),
        )
        axs = np.atleast_2d(axs)

        def animate_diff(i, store):
            plots = []
            frame = -store[i]
            vmin, vmax = frame.min(), frame.max()
            idx = 0
            for r in range(axs.shape[0]):
                for c in range(axs.shape[1]):
                    if idx >= n_sample: break
                    axs[r, c].clear()
                    axs[r, c].set_xticks([]); axs[r, c].set_yticks([])
                    plots.append(axs[r, c].imshow(frame[idx, 0], cmap="gray", vmin=vmin, vmax=vmax))
                    idx += 1
            return plots

        ani = FuncAnimation(fig, animate_diff, fargs=[x_gen_store], interval=200,
                            blit=False, repeat=True, frames=x_gen_store.shape[0])
        gif_path = f"{gif_prefix}_w{w}.gif"
        ani.save(gif_path, dpi=100, writer=PillowWriter(fps=5))
        plt.close(fig)
        print(f"saved final gif at {gif_path}")


# ----------------------------------------------------------------------
# Main training (unnormalized data, WaveguideDatasetTop1, gamma-blended loss)
# ----------------------------------------------------------------------
def train_waveguide_ddpm(
    h5_path='new_top1_cnn.h5',
    save_dir="./data/diffusion_new_top1/",
    n_epoch=20,
    batch_size=256,
    n_T=400,
    n_feat=128,
    lrate=1e-4,
    drop_prob=0.1,
    betas=(1e-4, 0.02),
    ws_test=(0.0, 0.5, 2.0),
    save_model=True,
    test_eval_fraction=0.25,
    device=None,
    cnn=None,        # optional (required if gamma < 1.0)
    ddpm_state=None,
    gamma: float = 0.5,   # 1.0 -> image-only; 0.0 -> CNN-target-only
):
    """
    Assumes:
      - Dataset class: WaveguideDatasetTop1(h5_path, split) -> (cond:[2], params:[P], pattern:[1,32,32])
      - ddpm.nn_model(x_t, cond, params, t_norm) accepts cond:[B,2], params:[B,P]
      - ddpm.sample(..., top_pair=cond, params=params, ...) uses same shapes
    """
    assert 0.0 <= gamma <= 1.0, "gamma must be in [0,1]"
    if gamma < 1.0:
        assert cnn is not None, "gamma < 1.0 requires a CNN to compare against dataset (neff, weight) targets."

    os.makedirs(save_dir, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    on_cuda = (device.startswith("cuda") and torch.cuda.is_available())

    # ---- Data ----
    # Import your dataset class before calling this function, or place it in the same file.
    train_ds = WaveguideDatasetTop1(h5_path, split="train")
    test_ds  = WaveguideDatasetTop1(h5_path, split="test")

    # h5py-friendly defaults first; increase workers later if needed
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=on_cuda)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=on_cuda)

    # ---- Model ----
    unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
    ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)

    if ddpm_state is not None:
        ddpm.load_state_dict(torch.load(ddpm_state, map_location=device))
        print(f"[resume] loaded DDPM from {ddpm_state}")

    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    # Freeze CNN (if used)
    if cnn is not None:
        cnn = cnn.to(device).eval()
        for p in cnn.parameters():
            p.requires_grad_(False)

    # ---- Tracking ----
    train_losses, test_losses = [], []
    best_test_loss = float("inf")

    # ---- Training loop ----
    for ep in range(n_epoch):
        print(f"epoch {ep}")
        ddpm.train()

        # linear LR decay
        for g in optim.param_groups:
            g["lr"] = lrate * (1 - ep / n_epoch)

        pbar = tqdm(train_loader)
        loss_ema = None
        train_loss_sum, train_loss_batches = 0.0, 0

        for cond, params, x_real in pbar:
            x_real = x_real.to(device, non_blocking=on_cuda)  # [B,1,32,32]
            cond   = cond.to(device, non_blocking=on_cuda)    # [B,2]
            params = params.to(device, non_blocking=on_cuda)  # [B,P]

            # random timestep
            B = x_real.size(0)
            t    = torch.randint(1, n_T + 1, (B,), device=device)
            noise = torch.randn_like(x_real)
            x_t   = ddpm.sqrtab[t, None, None, None] * x_real + ddpm.sqrtmab[t, None, None, None] * noise
            t_norm = (t.float() / n_T).unsqueeze(1)

            # classifier-free guidance dropout on (cond, params)
            context_mask = torch.bernoulli(torch.full((B,), ddpm.drop_prob, device=device))
            ctx = context_mask.unsqueeze(1)
            cond_m   = cond   * (1.0 - ctx)
            params_m = params * (1.0 - ctx)

            # predict noise and reconstruct x0
            optim.zero_grad(set_to_none=True)
            pred_noise = ddpm.nn_model(x_t, cond_m, params_m, t_norm)
            x0_hat = (x_t - ddpm.sqrtmab[t, None, None, None] * pred_noise) / ddpm.sqrtab[t, None, None, None]
            x0_hat = x0_hat.clamp(0.0, 1.0)

            # blended loss
            loss_img = F.mse_loss(x0_hat, x_real)
            if gamma < 1.0:
                pred_top = cnn(x0_hat, params)  # [B,2]
                targ_top = cond                 # dataset ground-truth top1 (neff, weight)
                loss_cnn = F.mse_loss(pred_top, targ_top)
            else:
                loss_cnn = x_real.new_tensor(0.0)

            loss = gamma * loss_img + (1.0 - gamma) * loss_cnn
            loss.backward()
            optim.step()

            # logging
            lv = float(loss.detach().item())
            train_loss_sum += lv
            train_loss_batches += 1
            loss_ema = lv if loss_ema is None else (0.95 * loss_ema + 0.05 * lv)
            pbar.set_description(f"loss:{loss_ema:.5f}  img:{loss_img.item():.3e}  cnn:{loss_cnn.item():.3e}")

        # epoch means
        mean_train_loss = train_loss_sum / max(train_loss_batches, 1)
        train_losses.append(mean_train_loss)

        # eval on subset
        mean_test_loss = evaluate_on_test_portion(
            ddpm=ddpm,
            test_loader=test_loader,
            n_T=n_T,
            device=device,
            cnn=cnn,
            test_eval_fraction=test_eval_fraction,
            clamp_for_cnn=True,
            gamma=gamma,
        )
        test_losses.append(mean_test_loss)
        print(f"epoch {ep}: train={mean_train_loss:.6f} | test={mean_test_loss:.6f}")

        # save best
        if save_model and mean_test_loss < best_test_loss:
            best_test_loss = mean_test_loss
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(ddpm.state_dict(), best_path)
            print(f"✔ improved test loss; saved best model to {best_path}")

        # final-epoch GIFs only
        if ep == n_epoch - 1:
            prefix = os.path.join(save_dir, "gif_final")
            save_final_gifs(
                ddpm=ddpm,
                test_loader=test_loader,
                test_ds=test_ds,
                device=device,
                ws_test=ws_test,
                max_samples=32,
                gif_prefix=prefix,
            )

        # plot losses each epoch
        try:
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(train_losses)+1), train_losses, label="Train (blended)")
            plt.plot(range(1, len(test_losses)+1),  test_losses,  label="Test (blended, subset)")
            plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"Train vs Test (gamma={gamma})")
            plt.legend(); plt.tight_layout()
            loss_curve_path = os.path.join(save_dir, "loss_curve.png")
            plt.savefig(loss_curve_path, dpi=150)
            plt.close()
            print(f"updated loss curve at {loss_curve_path}")
        except Exception as e:
            print(f"warning: failed to plot loss curve ({e})")

    # save final
    if save_model:
        final_path = os.path.join(save_dir, "model_final.pth")
        torch.save(ddpm.state_dict(), final_path)
        print(f"saved final model at {final_path}")


In [35]:
if __name__ == "__main__":
    train_waveguide_ddpm(cnn=None, gamma=1.0)
    train_waveguide_ddpm(cnn=cnn, gamma=0.75, save_dir="./data/diffusion_new_top1_gamma75/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.5, save_dir="./data/diffusion_new_top1_gamma50/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.25, save_dir="./data/diffusion_new_top1_gamma25/")

epoch 0


loss:0.25046  img:1.619e-01  cnn:0.000e+00:   0%|          | 7/4352 [00:00<08:42,  8.32it/s]  


KeyboardInterrupt: 

In [39]:
#!/usr/bin/env python3
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# import your dataset + model defs
from waveguide_dataset_top1 import WaveguideDatasetTop1

@torch.no_grad()
def compare_waveguides_3gens(
    ddpm,
    test_loader,
    device,
    guide_w: float = 2.0,
    n_rows: int = 12,
    save_path: str | None = None,
    binarize: bool = False,
    thresh: float = 0.5,
):
    """
    For the first n_rows items, generate 3 samples per condition and plot:
        Real | Gen1 | Gen2 | Gen3
    Expects loader to yield (cond:[B,2], params:[B,P], x_real:[B,1,32,32]).
    """
    ddpm.eval()

    # Grab one test batch
    try:
        cond, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    n = min(n_rows, cond.shape[0])
    cond   = cond[:n].to(device)      # [n,2]  (neff, weight)
    params = params[:n].to(device)    # [n,P]
    x_real = x_real[:n].to(device)    # [n,1,32,32]

    # Repeat each condition 3× to get 3 generated samples per item
    num_gens      = 3
    cond_rep      = cond.repeat_interleave(num_gens, dim=0)     # [n*3,2]
    params_rep    = params.repeat_interleave(num_gens, dim=0)   # [n*3,P]

    # Sample (DDPM randomness yields different outputs per repeat)
    x_gen_all, _ = ddpm.sample(
        n_sample=n * num_gens,
        size=(1, 32, 32),
        device=device,
        top_pair=cond_rep,     # [B,2]
        params=params_rep,     # [B,P]
        guide_w=guide_w,
    )  # [n*3,1,32,32]

    # Reshape to [n, 3, 1, 32, 32]
    x_gen_all = x_gen_all.view(n, num_gens, 1, 32, 32)

    # To numpy
    real_np = x_real.detach().cpu().numpy()          # [n,1,32,32]
    gen_np  = x_gen_all.detach().cpu().numpy()       # [n,3,1,32,32]

    real_np = np.clip(real_np, 0.0, 1.0)
    gen_np  = np.clip(gen_np,  0.0, 1.0)
    if binarize:
        real_np = (real_np >= thresh).astype(np.float32)
        gen_np  = (gen_np  >= thresh).astype(np.float32)

    # Plot: 4 columns => Real + 3 gens
    cols = 4
    fig_h = max(2, n * 1.1)
    fig, axs = plt.subplots(n, cols, figsize=(cols * 2.2, fig_h), squeeze=False)

    for i in range(n):
        # Real (red)
        ax = axs[i, 0]
        ax.imshow(real_np[i, 0], cmap="Reds", vmin=0, vmax=1)
        if i == 0: ax.set_title("Real", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # Gen 1..3 (black). Use 1 - gen for black-on-white with 'binary_r'
        for j in range(num_gens):
            ax = axs[i, j + 1]
            ax.imshow(1.0 - gen_np[i, j, 0], cmap="binary_r", vmin=0, vmax=1)
            if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()


if __name__ == "__main__":
    # ---- Paths / config ----
    h5_path   = "new_top1_cnn.h5"  # your new dataset
    save_root = "./data/diffusion_new_top1_gamma75/"
    model_ckpt = os.path.join(save_root, "best_model.pth")  # load this DDPM

    n_rows_to_show = 12
    guide_w = 2.0
    out_png = os.path.join(save_root, "compare_3gens.png")

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    on_cuda = torch.cuda.is_available()

    # ---- Dataset / DataLoader ----
    # Try true test split; if not present, fall back to train split.
    try:
        test_ds = WaveguideDatasetTop1(h5_path, split="test")
    except KeyError:
        print("[WARN] '*_test' datasets not found; using 'train' split for visualization.")
        test_ds = WaveguideDatasetTop1(h5_path, split="train")

    test_loader = DataLoader(
        test_ds,
        batch_size=64,              # just for visualization; not memory-heavy
        shuffle=False,
        num_workers=0,              # h5py-friendly default
        pin_memory=on_cuda,
    )

    # ---- Model ----
    unet = ContextUnet(in_channels=1, n_feat=128, use_time_embed=True)
    ddpm = DDPM(nn_model=unet, betas=(1e-4, 0.02), n_T=400, device=device, drop_prob=0.0).to(device)

    # Load trained weights
    ddpm.load_state_dict(torch.load(model_ckpt, map_location=torch.device(device)))
    ddpm.eval()
    print(f"Loaded DDPM weights from: {model_ckpt}")

    # ---- Compare real vs 3 generated per condition ----
    compare_waveguides_3gens(
        ddpm,
        test_loader,
        device=device,
        guide_w=guide_w,
        n_rows=n_rows_to_show,
        save_path=out_png,
        binarize=True,
        thresh=0.5,
    )


Loaded DDPM weights from: ./data/diffusion_new_top1_gamma75/best_model.pth
Saved comparison grid to ./data/diffusion_new_top1_gamma75/compare_3gens.png


In [ ]:
import h5py
with h5py.File("/home/omiqran/metamaterials_urop/new_top1_cnn.h5","r") as f:
    for k in ["pattern_train","params_train","neff_train","weight_train"]:
        d=f[k]; print(k, "compression=", d.compression, "chunks=", d.chunks, "dtype=", d.dtype)


pattern_train compression= None chunks= None dtype= int16
params_train compression= None chunks= None dtype= float64
neff_train compression= None chunks= None dtype= float64
weight_train compression= None chunks= None dtype= float64


In [ ]:
import h5py
with h5py.File("/home/omiqran/metamaterials_urop/train_test_split.h5","r") as f:
    for k in ["pattern_train","params_train","neff_train","weight_train"]:
        d=f[k]; print(k, "compression=", d.compression, "chunks=", d.chunks, "dtype=", d.dtype)


pattern_train compression= None chunks= None dtype= int16
params_train compression= None chunks= None dtype= float64
neff_train compression= None chunks= None dtype= float64
weight_train compression= None chunks= None dtype= float64
